## 一、推荐系统的评价指标
1. 北极星指标：用户规模、用户留存、消费；用户发布（小红书、抖音等用户生成内容(UGC)平台）；
2. DAU：日活用户数
3. MAU：月活用户数
4. WAU：周活用户数
5. FDAU：推荐日活用户数
6. SDAU：搜索日活用户数
7. retention rate：用户留存率——>次n日内留存率（次n留）/第n日留存率（第n留）
8. LT：用户生命周期 $LT_{n}=1+\sum_{i=1}^{n}r_{i}$， $r_{n}$为第n日留存
9. 人均阅读量：平均每天每位用户阅读了多少物品的详情页
10. 人均使用推荐时长
//
11. 非核心指标：点击率、有效点击率、交互指标（点赞、收藏、转发、关注、评论等）

## 二、A/B测试
### 1.实验组和对照组
对用户做随机分桶，比如分成10个桶，每桶占10%流量，取一组作为对照组，一个或多个桶作为实验组，计算每个实验组和对照组的差（diff），反应新策略对业务指标的影响，如果有一组实验结果显著正向，则可以扩大它的流量。
> 分桶必须足够均匀，保证各桶所有业务指标持平（至少精确到万分之一）。
### 2.分层实验
1. 目的：解决线上同时做数十个、甚至上百个AB测试的需求
2. 要求：同层互斥、不同层正交。例如分成：召回、粗排、精排、重排、前端界面等等，召回层和粗排层各自独立对用户做分桶，同层的桶交集为空（例如两个召回实验不会同时作用在一位用户上），不同层的桶正交（例如召回层的桶(10%)和粗排桶的桶(10%)的交集为1%）
### 3.holdout机制
把推荐系统看做一个层，跟用户界面层、广告层正交。把推荐层内部划分成10%vs90%互斥的两部分，其中10%用户的组作为holdout桶，90%用户用作推荐实验。

推荐层中90%的用户可以分成召回、粗排、精排、重排这样正交的层，每层都可以用全部90%的用户做实验。召回、粗排、精排、重排的有效实验均会带来跟holdout桶指标的diff，且实验的di会叠加（通常有折损）。如果公司以双月作为考核周期，那么每两个月结束的时候，计算90%实验流量和10%holdout流量各种指标的diff（需要做归一化），作为整个推荐部门双月对业务指标的贡献。

在每个考核周期结束之后，会清除holdout桶，也就是让推全的实验从90%用户扩大到100%的用户。然后会把用户随机划分为10%holdout桶vs90%实验桶，开始下一个考核周期。由于划分是随机的，新的holdout与实验桶的各种指标的di都几乎为0。随着召回、粗排、精排、重排的实验上线和推全，两个桶指标的di会逐渐扩大。
### 4.实验推全与反转实验
1. 实验推全：所有的实验都是从小流量开始，如果业务指标的diff显著正向，则推全实验。
2. 反转实验：每推全一个实验就新建一个推全层，它覆盖90%用户，新的推全层与召回等层正交。在推全层中，85%的用户使用新策略，5%的用户（反转桶）使用旧策略，这样就可以长期观测新策略与旧策略业务指标的diff。当考察周期结束，清除holdout桶时，将新策略应用到holdout桶的10%用户。当反转实验完成时，关闭实验，则新策略会应用到反转桶的5%的用户。

## 三、召回
### 1. ItemCF（基于物品的协同过滤）
1. 基于用户对物品的兴趣和物品之间的相似度预估用户对候选物品的兴趣
2. 物品相似度：两个物品的受众重合度越高，两个物品越相似。
$$
sim(i_1, i_2) = \frac{|V|}{\sqrt{|W_1|\cdot|W_2|}}
$$
其中 $V = W_1 \cap W_2$，$W_1$为喜欢物品$i_1$的用户，$W_2$为喜欢物品$i_2$的用户。
> 此公式没有考虑喜欢的程度：$like(user,item)$
$$
sim(i_1, i_2) = \frac{\sum_{v\in V}like(v,i_1)\cdot like(v,i_2)}{\sqrt{\sum_{u_1 \in W_1}like^{2}(u_1, i_1)}\cdot \sqrt{\sum_{u_2 \in W_2}like^{2}(u_2, i_2)}}
$$
> 实际上就是将不同用户看作不同维度，分为$i_1$和$i_2$两个向量，在不同维度上的值就是不同用户维度对两个物品的喜欢程度，将这两个向量做余弦相似度。
3. 预估用户对候选物品的兴趣：
$$
\sum_{j}like(user, item_{j}) \times sim(item_{j}, item) 
$$
4. 流程：事先做离线计算（建立“用户->物品”索引和“物品->物品”索引）——>线上做召回（用户近期感兴趣的物品列表last-n和last-n中每个物品的top-k相似物品），即最多nk个相似物品，用公式预估用户对物品的兴趣分数，返回分数最高的100个物品作为推荐结果。
### 2. Swing召回通道
1. 原理与ItemCF相同，唯一的区别在于物品相似度（考虑了重合的用户是否来自一个小圈子）
2. 物品相似度：
$$
sim(i_1, i_2) = \sum_{u_1 \in V}\sum_{u_2 \in V} \frac{1}{\alpha + \text{overlap}(u_1, u_2)}
$$
其中 $V = W_1 \cap W_2$，$W_1$为喜欢物品$i_1$的用户，$W_2$为喜欢物品$i_2$的用户。
$$
\text{overlap}(u_1, u_2) = |J_1 \cap J_2|
$$
其中 $J_1$为用户$u_1$喜欢的物品，$J_2$为用户$u_2$喜欢的物品。
### 3. UserCF（基于用户的协同过滤）
1. 基于用户之间的相似度和用户对物品的兴趣预估用户对候选物品的兴趣
2. 两个用户的相似度：
$$
sim(u_1, u_2) = \frac{|I|}{\sqrt{|J_1|\cdot |J_2|}}
$$
其中 $J_1$为用户$u_1$喜欢的物品，$J_2$为用户$u_2$喜欢的物品, $I = J_1 \cap J_2$.
> 此时不论冷门、热门，物品权重都是1，我们需要降低热门物品的权重。
$$
sim(u_1, u_2) = \frac{\sum_{l \in I}\frac{1}{\log(1+n_l)}}{\sqrt{|J_1|\cdot |J_2|}}
$$
其中 $n_l$为喜欢物品$l$的用户数量，反应物品的热门程度。

3. 预估用户对候选物品的兴趣：
$$
\sum_{j}sim(user, user_j) \times like(user_j, item)
$$
### 4. 离散特征处理
1. 建立字典：把类别映射成序号
2. 向量化：把序号映射成向量
- One-hot编码：把序号映射成高维稀疏向量
- Embedding：把序号映射成低维稠密向量
3. Embedding（嵌入）：参数数量=向量维度 $\times$ 类别数量，第i个序号对应参数矩阵的第i列
> Embedding = 参数矩阵 $\times$ One-Hot向量
### 5. 矩阵补充、最近邻查找
1. 基本想法：训练用户embedding参数矩阵和物品embedding参数矩阵，以用户embedding向量和物品embedding向量的内积表示用户对物品的兴趣分数以补充用户对未曝光物品的兴趣分数。
2. 数据集：（用户ID，物品ID，兴趣分数），记作 $\Omega = \{(u, i, y)\}$
3. 优化目标函数：
$$
\min_{A,B} \sum_{(u,i,y)\in \Omega}(y-<a_u, b_i>)^2
$$
4. 缺点：
- 仅用ID embedding，没利用物品、用户属性；
- 负样本选取方式不对
- 做训练的方法不好（内积不如余弦相似度）
5. 线上服务：把用户ID作为key，查询key-value表，得到用户向量，记作a；接着最近邻查找来查找用户最可能感兴趣的k个物品作为召回结果。
6. 近似最近邻查找：将所有物品在空间上分成多个区域（如扇形），每个区域都用一个单位索引向量表示，给定一个索引向量可以快速取回该区域的所有点，对用户向量a，先计算a与所有索引向量计算相似度，取相似度最大的索引向量，计算a与索引向量对应区域所有点的相似度，取最高的k个点作为召回结果。
### 6. 双塔模型：模型和训练
1. 模型：分别考虑用户和物品的ID、离散特征、连续特征，对ID和离散特征经Embedding层向量化，对连续特征归一化、分通等处理，对三者处理后的向量连接输入神经网络中输出用户和物品的表征向量a和b，作a和b的内积/余弦相似度。
> 召回双塔模型一定是用户塔和物品塔先特征变换、输入神经网络后分别输出一个向量再做内积（即后期融合），而不能前期特征变换后即连接输入一个神经网路（即前期融合）
> 原因：召回考虑物品数量太大，后期融合可以使用近似最近邻查找等快速最近邻（减少时间复杂度）的方法，而前期融合无法预先计算好所有物品的表示，要召回k个物品必须让用户特征和每个物品特征融合后输入神经网路，这样计算量非常大。
2. 训练：
- Pointwise：独立看待每个正样本、负样本，做简单二元分类
- Pairwise：每次去一个正样本、一个负样本
- Listwise：每次去一个正样本、多个负样本
3. 正负样本选择
- 正样本：用户点击的物品
- 负样本：没有被召回的？召回但是被粗排、精排淘汰的？曝光但是未点击的？
4. Pointwise训练：
- 把召回看作二元分类任务
- 控制正负样本数量1：2或1：3
- 正样本鼓励 $\cos(a,b)$ 接近+1，负样本鼓励 $\cos(a,b)$ 接近-1
5. Pairwise训练：
- Triplet hinge loss：
$$
L(a, b^{+}, b^{-}) = \max\{0, \cos(a,b^{-}) + m - \cos(a, b^{+})\}
$$
- Triplet logistic loss:
$$
L(a, b^{+}, b^{-}) = \log(1 + \exp[\sigma \cdot (\cos(a,b^{-})  - \cos(a, b^{+}))])
$$
6. Listwise训练:
- 将用户表征向量和一个正样本、多个负样本的余弦值经softmax激活，以1为正样本的标签，0为负样本标签，将激活后的s和标签y做交叉熵当作损失函数，即 $-\log s^{+}$。
### 7. 双塔模型：正负样本
1. 正样本：曝光而且有点击的用户-物品二元组
> 问题：正样本大多是热门物品
> 解决方案：过采样冷门物品，或降采样热门物品。
- 过采样（up-sampling）：一个样本出现多次
- 降采样（down-sampling）：一些样本被抛弃
2. 简单负样本：全体物品
- 均匀抽样：对冷门物品不公平
- 非均匀采样：打压热门物品；抽样概率与热门程度（点击次数）正相关；$\text{抽样概率}\propto(\text{点击次数})^{0.75}$
3. 简单负样本：Batch内负样本
4. 困难负样本：
- 被粗排淘汰的物品
- 精排分数靠后的物品
5. 训练数据：混合几种负样本
> 错误：选择曝光但未点击的物品作为负样本，这是错误的，不能当作召回的负样本（可以作为排序的样本）
6. 召回的目标：快速找到用户可能感兴趣的物品。
### 8. 双塔模型：线上召回和更新
1. 线上召回
- 完成训练后事先离线存储<物品特征向量b, 物品ID>保存到向量数据库以作最近邻查找；
- 对给定用户要线上现计算用户特征向量a，通过最近邻查找余弦相似度最大的k个物品作为召回结果
- 原因：用户兴趣动态变化，物品特征相对稳定
2. 模型更新
- 全量更新：今天凌晨，用昨天全体的数据训练模型。（在昨天凌晨模型参数基础上训练，不是昨天增量更新后的模型参数）
- 增量更新：做 online learning 更新模型参数。（实时收集线上数据做流式处理）（增量更新ID Embedding参数，不更新神经网络其他部分参数）
- 能否只做增量更新，不做全量更新？（不能，全量更新随机打乱一天的数据，优于按顺序排列数据，减小了偏差）
### 9. 双塔模型 + 自监督学习
1. 双塔模型的问题：
- 头部效应
- 长尾物品表征学得不好
- 自监督学习可以很好学习长尾物品表征
2. 自监督学习训练物品塔
- 相同物品不同特征变换后经物品塔的向量表征的相似度尽量大，不同物品向量表征的相似度尽量小。
- 特征变换-Random Mask：随机选一些离散特征遮住
- 特征变换-Dropoout：随机丢弃特征中50%的值（仅对多值离散特征）
- 特征变换-互补特征：随机把特征分成两组，对没有的特征记作default
- 特征变换-Mask一组关联的特征：离线计算特征两两之间的关联，用互信息衡量
$$
MI(U, V) = \sum_{u \in U} \sum_{v \in V} p(u, v) \cdot \log\frac{p(u, v)}{p(u)\cdotp(v)}
$$
3. 训练模型
- 从全体物品均匀抽样得到m个物品，作为一个batch
- 对物品做两种特征变换，物品塔输出两组向量
- 对第i个物品与batch内所有物品做余弦，经softmax激活，以相同物品的标签为1，其他为0，做交叉熵损失训练 
### 10. Deep Retrieval 召回
1. 索引
- 物品表示为路径（一个物品对应多条路径、一条路径对应多个物品）
2. 预估模型
- 预估用户(用户特征x)对路径的兴趣：$p(a, b, c |x)$
3. Beam Search（beam size=1, 2, 3, ...）
4. 线上召回：
- 给定用户特征，用神经网络做预估，用 beam search 召回一批路径
- 利用索引，根据召回的路径召回一批物品（每条路径对应多个物品）
- 对物品做排序，选出一个子集
5. 训练
- 同时学习神经网络参数和物品表征
- 如果用户点击过物品，则更新神经网络参数，使分数增大
- 如果用户对路径的兴趣分数较高，且用户点击过物品item，则item与path具有相关性。
- 寻找与item最相关的J条path，且避免一条路径上物品过多。
### 11. 其他召回通道
1. 地理位置召回（GeoHash召回/同城召回）
2. 作者召回（关注作者/有交互的作者/相似作者）
3. 缓存召回：复用前n次推荐精排的结果；需要退场机制
### 12. 曝光过滤 & Bloom Filter
1. Bloom Filter （k=？）
- Bloom Filter是一个m维的二进制向量，通过k个哈希函数判断物品是否曝光
2. Bloom filter 误伤的概率为：
$$
\delta \approx (1-\exp(-\frac{kn}{m}))^{k}
$$
-设定可容忍的误伤概率，可求得最优参数k和m
3. Bloom filter缺点
-只支持添加物品，不支持删除物品
-每天都需要从物品集合中移除年龄大于特定时长的物品，都需要重新计算Bloom filter

## 四、排序
### 1. 多目标模型
- 以用户特征、物品特征、统计特征、场景特征经concatenation作为输入输入一个神经网络，输出一个向量，分别经四个不同的全连接层+sigmoid，输出点击率、点赞率、收藏率、转发率。（前期融合）
- 训练：以交叉熵损失作为每个输出的损失，再使用加权和作为总的损失函数
- 困难：类别不平衡；解决方法：负样本降采样，抛弃一部分负样本（会导致预估点击率大于真实点击率）
- 预估值校准：校准公式：设 $\alpha$ 为采样率
$$
p_{true} = \frac{\alpha\cdot p_{pred}}{(1-p_{pred}) + \alpha\cdot p_{pred}}
$$
### 2. Multi-gate Mixture-of-Experts（MMoE）
- 特征向量不再输入一个神经网络，而是输入三个神经网络，同时特征向量输入另外两个神经网络，再经softmax激活出三个权重（2组，一个神经网络一组）
- 分别用两组权重对三个神经网络输出的向量做加权平均，分别输出点击率和点赞率，如果需要多个指标输出可以增加多个权重神经网络
- 极化现象：softmax输出值一个接近1，其余接近0
- 解决极化现象的方法：训练时，对softmax的输出使用dropout，例如10个“专家”被dropout的概率都是10%
### 3. 预估分数的融合
1. 简单加权和
2. 点击率乘以其他项的加权和
3. 等等
### 4. 视频播放建模
> 相比于图文笔记，排序依据还有播放时长和完播
1. 播放时长
- 特征向量经神经网络——>全连接层(输出实数)——>sigmoid变换——>与 $y=\frac{t}{1+t}$ 做交叉熵
$$
CE(y, p) = y\cdot \log p + (1-y) \cdot \log(1-p)
$$
> $ \exp (z)$ 作为对播放时长的预估
2. 完播
- 回归方法
- 二元分类
- 不能直接把预估的完播率用到融分公式，需要用函数 $f(\text{视频时长})$ 拟合完播率，然后做调整：
$$
p_{finish} = \frac{\text{预估完播率}}{f(\text{视频长度})}
$$
> 把 $p_{finish}$ 加入融分公式。
### 5. 排序模型的特征
1. 用户画像（User Profile）
- 用户ID（在召回、排序中做embedding）
- 人口统计学属性：年龄、性别
- 账号信息：新老、活跃度...
- 感兴趣的类目、关键词、品牌
2. 物品画像（Item Profile）
- 物品ID
- 发布时间
- GeoHash（经纬度坐标）、所在城市
- 标题、类目、关键词、品牌...
- 字数、图片书、视频清晰度、标签数...
- 内容信息量、图片美学（事先标注输入CV等模型打分）
3. 用户统计特征
- 用户最近30天（7天、1天、1小时）的曝光数、点击数、点赞数、收藏数...
- 按照笔记图文/视频分桶。（比如最近7天，该用户对图文笔记的点击率、对视频笔记的点击率）
- 按照笔记类目分桶。
4. 笔记统计特征
- 笔记最近30天（7天、1天、1小时）的曝光数、点击数、点赞数、收藏数
- 按照用户性别分桶、按照用户年龄分桶
- 作者特征：发布笔记数、粉丝数、消费指标（曝光数、点击数、点赞数、收藏数）
5. 场景特征（Context）
- 用户定位GeoHash、城市
- 当前时刻（分段，做embedding）、
- 是否是周末、是否是节假日
- 手机品牌、手机型号、操作系统
6. 特征处理
- 离散特征：做embedding
- 连续特征：做分桶，变成离散特征
- 连续特征：其他变换（曝光数、点击数、点赞数等数值做 $\log(1+x)$）（转化为点击率、点赞率等值，并做平滑）
7. 特征覆盖率
- 很多特征无法覆盖100%样本
- 提高特征覆盖率，可以让精排模型更准
### 6. 粗排模型
1. 三塔模型（用户塔、物品塔、交叉塔（统计特征、交叉特征））

## 五、特征交叉
### 1. Factorized Machine（FM）
$$
p = b + \sum_{i=1}^{d} w_{i}x_{i} + \sum_{i=1}^{d}\sum_{j=i+1}^{d}(v_{i}^{T}v_{j})x_{i}x_{j}
$$
其中 $k \ll d$.
### 2. 深度交叉网络（DCN）
> 可用于召回、排序
- 交叉层（Cross Layer）：$x_{i+1} = x_{0} \odot [W\cdot x_{i} + b] + x_{i}$
- 交叉网络（Cross Network）
- 深度交叉网络：特征拼接后同时输入全连接网络和交叉网络（并联），将两个输出向量拼接输入全连接层，再输出。
### 3. LHUC网络结构
> 只能用于精排

>用户特征输入的神经网络最后输出要用sigmoid函数乘以2

![LHUC](images/LHUC.png)
### 4. SENet
![SENet](images/SENet.png)
> 不同行的列数（k）可以不同

> 对离散特征做 field-wise 加权

> Field：如用户ID Embedding 是64维向量，则64个元素算一个field，获得相同权重

> 如果有 m 个 fields，那么权重向量是 m 维。
### 5. Field间特征交叉
1. 内积
2. 哈达玛乘积
3. Bilinear Cross（内积）
$$
f_{ij} = x_{i}^{T} \cdot W_{ij} \cdot x_{j}
$$
4. Bilinear Cross（哈达玛乘积）
$$
f_{ij} = x_{i} \odot [W_{ij} \cdot x_{j}]
$$
### 6. FiBiNet
![FiBiNet](images/FiBiNet.png)

## 六、行为序列（用户行为序列建模）
### 1. LastN特征
- LastN：用户最近n次交互的物品ID（可以是点击、点赞、收藏、转发等都分别做最后拼起来）
- 对LastN物品ID做embedding，得到n个向量（也可以用物品的其他特征做embedding最后拼起来）
- 把n个向量取平均，作为用户的一种特征
- 适用于召回双塔模型、粗排三塔模型、精排模型
### 2. DIN模型（注意力机制）
> 适用于精排模型
- DIN 用加权平均代替平均，即注意力机制（attention）
- 权重：候选物品与用户LastN物品的相似度
- 对于某候选物品，计算它与LastN的相似度，以相似度为权重求LastN的加权和向量作为用户特征输入排序模型
- 本质是注意力机制 
- 缺点：计算量 $\propto n$，只能记录最近几百个物品，遗忘长期兴趣
### 3. SIM模型
- 改进DIN：快速排除掉与候选物品无关的LastN物品，降低注意力层的计算量
- 做法：对于每个候选物品，在用户LastN中做快速查找，找到k个相似物品，把LastN变成TopK，然后使用时间信息拼接，最后输入注意力层。
1. Hard Search：根据候选物品类目，保留类目相同的
2. Soft Search：把物品做embedding，变成向量，把候选物品向量作为query，做k近邻查找，保留LastN中最接近的k个
3. 使用时间信息：用户与某个LastN的交互时刻距今为 $\delta$，对 $\delta$ 做离散化，在做embedding，变成向量 $d$，接着与LastN物品embedding拼接
> 优点：长序列、注意力机制、时间信息

## 七、重排
### 1. 物品相似性的度量
1. 基于物品属性标签
- 类目、品牌、关键词
- 根据一级类目、二级类目、品牌计算相似度
2. 基于物品向量表征
- 用召回的双塔模型学到的物品向量（不好）
- 基于内容的向量表征
3. CLIP预训练
- 图片-文字二元组，同一个二元组的两个向量要尽可能相似，可以取一个batch，原理同batch内负样本
4. 粗排、精排后处理
- 用于增加多样性，从n个候选物品中选出k个，既要总分高，也要多样性。
- 精排的后处理被称为 “重排”。
- 需要多样性算法
### 2. Maximal Marginal Relevance（MMR）多样性算法
1. 原理
- 将n个物品分成选中的物品集合$S$和未选中的物品集合 $R$，计算 $R$ 中每个物品 $i$ 的 Marginal Relevance 分数：
$$
\text{MR}_i = \theta \cdot \text{reward}_i - (1-\theta) \cdot \max_{j \in S} \text{sim}(i, j)
$$
- MMR:
$$
\argmax_{i\in R} \text{MR}_i
$$
2. 流程
- 已选中的物品集合初始化为空集，未选中的物品集合初始化为全集
- 选择精排分数最高的物品，从集合 $R$ 移到 $S$
- 做k-1轮循环：计算MR分数，选出最高的移动
- 缺点：已选中的物品越多，越难找出物品使得该物品与已选中的物品都不相似
3. 滑动窗口
- 设置一个滑动窗口，比如最近选中的10个物品，用这个滑动窗口代替MMR中的S
$$
\argmax_{i\in R} \{\theta \cdot \text{reward}_i - (1-\theta) \cdot \max_{j \in W} \text{sim}(i, j)\}
$$
- 实际工业界应用时都要带滑动窗口
### 3. 重排的规则
1. 最多连续出现k篇某种笔记
2. 每k篇笔记最多出现1篇某种笔记
> 例如运营推广笔记
3. 前t篇笔记最多出现k篇某种笔记
4. 等等规则
### 4. DPP多样性算法的数学基础
1. 超平行体
- 2维的超平行体为平行四边形
- 3维的超平行体为平行六面体
- 一组不相关向量 $v_1, \cdots, v_k \in \mathbb{R}^d$ 可以确定一个 $k$ 维 ($k\leq d$) 超平行体：
$$
\mathbf{P}(v_1, \cdots, v_k) = \{\alpha_1 v_1 + \cdots + \alpha_k v_k | 0 \leq \alpha_1, \cdots, \alpha_k \leq 1\}
$$
- 如果 $v_1, \cdots, v_k$ 线性相关，则体积 $\text{vol}(\mathbf{P}) = 0$
2. 衡量物品多样性
- 给定k个物品，把它们表征为k个d维单位向量（$d \geq k$）
- 用超平形体的体积衡量物品的多样性（0~1）
- 如果单位向量两两正交，则体积最大化为1
- 如果单位向量线性相关，则体积最小化为0
3. 行列式与体积的关系
- 对于 $v_1, \cdots, v_k \in \mathbb{R}^d$, 如果 $k=d$, 则：
$$
\det(V) = \text{vol}(\mathbf{P}(v_1, \cdots, v_k))
$$
- 对于 $v_1, \cdots, v_k \in \mathbb{R}^d$, 如果 $k<d$, 则：
$$
\det(V^T V) = \text{vol}(\mathbf{P}(v_1, \cdots, v_k))^2
$$
### 5. DPP多样性算法
1. 行列式点过程（DPP）
- DPP是一种传统的统计机器学习方法：
$$
\argmax_{S:|S|=K} \log \det (V_{S}^{T}V_{S})
$$
2. DPP应用到推荐系统：
$$
\argmax_{S:|S|=K} \theta\cdot(\sum_{j\in S} \text{reward}_{j}) + (1-\theta)\cdot\log \det (V_{S}^{T}V_{S})
$$
3. 求解方式
- 记 $A_S = V_{S}^{T}V_{S}$ 
- 设 $S$ 为已选中的物品，$R$ 为未选中的物品，用贪心算法求解：
$$
\argmax_{i \in R} \theta\cdot\text{reward}_{i} + (1-\theta)\cdot\log \det (A_{S\cup \{i\}})
$$
- 暴力算法的总时间复杂度为：$O(n^2 d + nk^4)$
4. Hulu 的快速算法
- Cholesky分解可供计算矩阵的行列式
- 给矩阵增加一行和一列可以快速计算出新矩阵的Cholesky分解
5. 滑动窗口
$$
\argmax_{i \in R} \theta\cdot\text{reward}_{i} + (1-\theta)\cdot\log \det (A_{W\cup \{i\}})
$$
6. 规则约束
- 用规则排除掉R中的部分物品，得到子集，在子集中选择下一个物品。

## 八、物品冷启动
### 1. 优化目标&评价指标
1. 目标
- 精准推荐
- 激励发布
- 挖掘高潜：通过初期小流量试探
2. 评价指标
- 作者侧指标
- 用户侧指标：新笔记指标+大盘指标（冷启动目的不是提高消费指标，但也不能显著降低）
- 内容侧指标
3. 作者侧指标
- 发布渗透率 = 当日发布人数/日活人数
- 人均发布量 = 当日发布笔记数/日活人数
> 反映作者的发布积极性
4. 用户侧指标
- 新笔记消费者指标：点击率、交互率（分别考虑高曝光、低曝光新笔记）
- 大盘的消费指标（消费时长、日活、月活）
5. 内容侧指标
- 高热笔记占比
### 2. 物品冷启动：简单的召回通道
1. 难点
- 缺少用户交互，没学好笔记 ID Embedding，导致双塔模型效果不好（改造后适用）
- 缺少用户交互，导致ItemCF效果不好（物品相似度依赖与物品交互的用户）
2. 改造双塔模型
- 法一：新物品 ID embedding 适用 default embedding（共享一个）
- 法二： 相似笔记 embedding
- 实践中通常使用多个向量召回池（不同时间）
3. 基于类目的召回
4. 基于关键词的召回
> 3、4缺点：只对刚刚发布的新笔记有效且弱个性化
### 3. 聚类召回
1. 思想
- 事先训练一个神经网络，基于笔记类目和图文内容，把笔记映射到向量
- 对笔记向量做聚类，划分为 1000 cluster，记录每个cluster的中心方向
2. 聚类索引
- 新笔记用神经网络映射到一个特征向量，从100个cluster中找到最相似的向量作为新笔记的cluster
- 做cluster-> 笔记ID列表（按时间倒排）的索引
3. 线上召回
- 给定用户ID，找到last-n交互
- 把每个last-n映射到向量，寻找最相似的cluster
- 每个cluster索引中取回m篇笔记
4. 内容相似度模型训练
- （正样本笔记，种子笔记，负样本笔记）类似双塔模型Pairwise训练
- 正样本选取：1.人工标注；2.算法自动选正样本（筛选条件：只用高曝光且有相同的二级类目，用ItemCF相似度）
- 负样本选取：从全体笔记随机选，满足：字数较多且笔记质量高，避免图文无关
### 4. Look-Alike人群扩散
1. 种子用户（对新笔记有交互行为的用户）
- 对一个新笔记的种子用户的特征向量平均为**笔记的特征向量** ，每当有用户交互更新笔记的特征向量（近线即可）
2. 召回
- 每当用户刷新，用双塔模型得到用户的特征向量，在新笔记特征向量数据库中适用最近邻查找10篇笔记
### 5. 流量调控
> 把流量向新笔记倾向以促进发布和挖掘优质笔记
1. 新笔记提权
- 设置提权系数
- 缺点：曝光对提权系数很敏感，很难控制
2. 新笔记保量
- 在原有提权系数基础上，乘以额外的提权系数
- 额外提权系数受发布时间和曝光次数影响
3. 动态提权保量
$$
\text{提权系数} = f(\frac{\text{发布时间}}{\text{目标时间}}, \frac{\text{已有曝光}}{\text{目标曝光}})
$$
4. 保量的难点
- 保量成功率远低于100%
- 线上环境变化会导致保量失败
5. 差异化保量
- 不同笔记有不同保量目标
- 基础保量 + 内容质量 + 作者质量
### 6. 冷启动的AB测试
1. 用户侧实验：将用户分为实验组和对照组，分别用新策略和旧策略从全体笔记中做推荐
> 新策略和旧策略的diff在推全后可能缩小，因为新笔记保量一定，新策略变多，旧策略就会变少，diff比全用新策略增大
2. 作者侧实验
- 方案一：只分作者；缺点：新笔记两个桶抢流量导致结果不可信
- 方案二：分作者和用户，实验组对照组分别对应；缺点：新笔记池减少一半，对用户体验造成负面影响
- 方案一二相同缺点：新笔记和老笔记抢流量，结果与推全结果有差异
- 方案三，老笔记、用户侧、作者侧全部分开，分别对应；缺点：内容池小一半，用户消费指标大跌，损失业务

## 九、推荐系统涨指标的方法
### 1. 推荐系统的评价指标
1. 日活用户数（DAU）和留存是最核心的指标
2. 最常用 LT7 和 LT30 衡量留存
- LT7： t0登录，未来7天（t0~t6）中有4天登录，那么今天（t0）的LT7等于4.
3. 其他核心指标：用户使用时长、总阅读数、总曝光数等
### 2. 改进召回
> 双塔和item-to-item是最重要的两类召回模型，占据召回的大部分配额
1. 双塔模型
- 方向1：优化正样本、负样本（简单正样本、简单负样本、困难负样本）
- 方向2：改进神经网络结构（DCN代替全连接网络；用户塔中使用用户行为序列；多向量模型代替单向量模型）
- 方向3：改进模型的训练方法（结合二分类、batch内负采样；使用自监督学习）
2. Item-to Item（I2I）
> 基于相似物品做召回

> 最常见用法是U2I2I
- 方法1：ItemCF及其变体
- 方法2：基于物品向量表征计算物品相似度
3. 小众的召回模型
- U2U2I
- U2A2I
- U2A2A2I
### 3. 改进排序模型
1. 精排模型：基座
- 基座输入：输入包括离散特征和连续特征，输出一个向量，作为多目标预估的输入
- 改进1：基座加宽加深，计算量更大，预测更准确
- 改进2：做自动的特征交叉
- 改进3：特征工程，比如添加统计特征、多模态内容特征
2. 精排模型：多目标预估
- 改进1：增加新的预估目标，并把预估结果加入融合公式
- 改进2：MMoE、PLE
- 改进3：纠正 position bias
3. 粗排模型的改进
> 打分量比精排大10倍，必须够快
- 简单模型：多向量双塔模型
- 复杂模型：三塔模型
4. 粗精排一致性建模
- 蒸馏精排训练粗排，让粗排与精排更一致
- 方法1：pointwise蒸馏（以用户真实行为和精排预估两者的平均值作为粗排拟合的目标）
- 方法2：pairwise或listwise蒸馏（给定候选物品，按照精排预估做排序，做learning to rank（LTR），让粗排拟合物品的序（而非值））
- 缺点：精排出bug，会污染粗排
5. 用户行为序列建模
- 改进1：增加序列长度，但会增加计算成本和推理时间
- 改进2：筛选的方法，比如类目、物品向量表征聚类
- 改进3：对用户行为序列中的物品，使用ID以外的一些特征
6. 在线学习
- 线上m个模型，其中1个是holdout，1个是推全的模型，m-2个测试的新模型
7. 老汤模型
- 用每天新产生的数据对模型做1epoch的训练，久而久之老模型训练得非常好
- 对于新老模型都随机初始化全连接层，embedding层可以随机初始化也可以服用老模型参数，用n天数据训练新老模型做对比来快速判断新模型结构是否优于老模型
- 如何更快追平线上老模型？
- 方法1：尽可能多服用老模型训练好的embedding层，避免随机初始化
- 方法2：用老模型做teacher，蒸馏新模型
### 4. 多样性
> 精排用滑动窗口，粗排不用
- 粗排从5000中选500，可以先根据兴趣分数选200，再加入多样性分数选300
- 双塔模型：加入噪声（添加噪声会增加多样性）
- 双塔模型：抽样用户行为序列（最近交互n个物品，线保留最近r个，再随机在剩下抽样t个（t远小于n））
- U2I2I：抽样用户行为序列（做非均匀抽样）
- 探索流量：维护一个精选内容池，可以分人群，从中随机抽样几个物品跳过排序直接插入最终排序结果
### 5. 针对特殊用户人群
1. 构造特殊内容池，用于召回
- 新用户、低活用户行为少，个性化召回不精确
- 方法1：根据物品获得得交互次数、交互率选择优质物品（圈定人群构造内容池）
- 方法2：做因果推断，判断物品对人群留存率得贡献
- 通常使用双塔模型从特殊内容池做召回
2. 使用特殊排序策略
- 排除低质量物品（对于新用户、低活用户只关注留存，少出广告；新发布物品不在新用户、低活用户上做探索）
- 差异化的融分公式（低活用户提高预估点击率权重，保留曝光坑位给预估点击率最高的几个物品）
3. 特殊的排序模型
- 方法1：大模型+小模型（全体用户行为训练大模型，用特殊用户的行为训练小模型，小模型预估拟合大模型的残差）（小模型起到纠偏的作用）
- 方法2：融合多个experts，类似MMoE（与MMoE区别：根据用户特征计算权重）
- 方法3：大模型预估之后，小模型做校准（小模型输入：用户特征、大模型预估的结果；输出：拟合用户真实行为）
### 6. 利用交互行为
1. 关注
- 用户留存率与他关注的作者数量正相关
- 方法1：用排序策略提升关注量（认为定义关于用户关注作者数量的单调递减函数乘在预估关注率上加入融分公式）
- 方法2：构造促关注内容池和召回通道（仅在用户关注作者数较少时起作用）
- 交互（尤其是关注、评论）可以提升作者发布积极性（UGC平台将作者发布量、发布率作为核心指标）
- 用排序策略帮助低粉新作者涨粉（定义关于作者粉丝数的反调递减函数乘到该作者某作品推荐给某用户的预估关注率上加入融分公式）
- 隐式关注关系：用户u喜欢作者a发布的物品，但是u没有关注a，需要挖掘饮食关注关系
2. 转发
- 促转发（分享回流）
- KOL（Key Opinion Leader）建模：（利用历史转发带来的站外流量）识别站外KOL之后用于排序和召回
- KOL用于排序和召回方法1：添加额外一项 $k_u \cdot p_{ui}$，如果用户u是站外KOL，则 $k_u$ 大
- KOL用于排序和召回方法2：构造促转发内容池和召回通道
3. 评论
- 排序融分公式额外添加一项 $w_i \cdot p_i$， $w_i$与评论数负相关
- 喜欢评论的用户添加促评论内容池
- 常留高质量评论（点赞数高）的用户，用排序和召回策略鼓励这些用户多留评论（例如更容易被展示、置顶、被点赞、高曝光等）